# MedQuAD QLoRA training
This notebook only orchestrates the Python scripts staged in the private Kaggle dataset. Enable a GPU and internet in Kaggle settings.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import os, shutil, sys, tarfile
input_root = Path('/kaggle/input/llama-finetuning-code')
project = Path('/kaggle/working/medical-project')
archive_path = input_root / 'project.tar'
if archive_path.is_file():
    project.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as archive:
        archive.extractall(project)
elif (input_root / 'src/train.py').is_file() and (input_root / 'configs/config.yaml').is_file():
    for name in ('src', 'configs', 'data', 'requirements.txt', 'requirements-ci.txt'):
        source = input_root / name
        destination = project / name
        if source.is_dir():
            shutil.copytree(source, destination, dirs_exist_ok=True)
        elif source.is_file():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
else:
    available = sorted(str(path.relative_to(input_root)) for path in input_root.rglob('*') if path.is_file()) if input_root.is_dir() else []
    raise FileNotFoundError(f'Code dataset missing or incomplete at {input_root}. Attached files: {available[:20]}')
os.chdir(project)
sys.path.insert(0, str(project))
print('Project:', project)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
def run(module):
    subprocess.run([sys.executable, '-m', module], cwd=project, check=True)
run('src.prepare_data')


In [ ]:
run('src.train')
run('src.analyze_training')


In [ ]:
run('src.evaluate')
run('src.report')


In [ ]:
import yaml
config = yaml.safe_load((project / 'configs/config.yaml').read_text())
if config['publishing']['enabled']:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    run('src.upload_adapter')
else:
    print('Publishing disabled; adapter retained as Kaggle output')
